## Account Plan Creation


In [ ]:
import enum
from pydantic import BaseModel
import requests
import json
import re
from typing import List, Optional
from echo.tools.web_scraping import extract_data_from_links
from echo import sqldb

# Replace with your actual port if different from 3000
API_URL = "http://localhost:3000/api/search"

sqldb.create_table(
    '''CREATE TABLE IF NOT EXISTS search_results (
        company_name TEXT,
        query_type TEXT,
        query TEXT,
        message TEXT,
        sources TEXT,
        source_extracted_data TEXT DEFAULT NULL,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (company_name, query_type)
    );'''   
)


class Metadata(BaseModel):
    title: str
    url: str

class Source(BaseModel):
    pageContent: str
    metadata: Metadata


class ExtractedData(BaseModel):
    link: str
    data: str


class CitedSource(BaseModel):
    citation_id: int
    title: str
    content: str
    url: str
    

class ExtractedCitedSource(CitedSource):
    data: Optional[str] = None
    

class SearchResponse(BaseModel):
    message: str
    sources: List[Source]
    company_name: str
    seller: str
    query_type: str
    query: str
    source_extracted_data: Optional[List[ExtractedCitedSource]] = None
    timestamp: Optional[str] = None


class SearchResult(BaseModel):
    message: str
    cited_content: List[str]
    


class QueryTypes(enum.Enum):
    FMOD = "Financial Moddeling"
    STRATEGY = "Strategic Initiatives"
    COMPANALYSIS = "Competitor Analysis"
    RECENTNEWS = "Recent News & Events"


query_type_prompts = {
    QueryTypes.FMOD.value: {
        "search": "Gather financial information including size, industry, and revenue about the company - {company_name}",
        "system": (
            "You are a financial analyst. "
            "You need to gather financial information about the company that is relevant to the user query. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {company_name}:\n{company_info} "
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            "Extract the financial information like revenue, growth plan, profit about the company."
        )
    },
    QueryTypes.STRATEGY.value: {
        "search": (
            "Identify the company's key business priorities and strategic initiatives for the company - {company_name}. "
            "Analyze 10-K reports and financial statements. "
            "Gain insights into the company's operations, products, services, and market position. "
            "Assess the company's revenue, profitability, and overall financial stability to gauge its potential as a client."
            "Understand the company's future plans and priorities. "
            "Identify challenges the company faces, enabling you to position your product or service as a solution to mitigate these risks. "
        ),
        "system": (
            "You are a strategic analyst. "
            "You need to gather strategic initiatives about the company that is relevant to the user query. "
            "Find the information from annual reports and financial statements of the company. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {company_name}:\n{company_info} "
            
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            
            "Identify the company's key business priorities and strategic initiatives for the company - {company_name}. "
            "Analyze 10-K reports and financial statements. "
            "Gain insights into the company's operations, products, services, and market position. "
            "Assess the company's revenue, profitability, and overall financial stability to gauge its potential as a client."
            "Understand the company's future plans and priorities. "
            "Identify challenges the company faces, enabling you to position your product or service as a solution to mitigate these risks. "
        )
    },
    QueryTypes.RECENTNEWS.value: {
        "search": (
            "Gather recent news and events about the company - {company_name}. "
            "Identify any recent developments, announcements, or changes that may impact the company's operations or strategy. "
            "This information is crucial for understanding the company's current position and future outlook."
        ),
        "system": (
            "You are a news analyst. "
            "You need to gather recent news and events about the company that is relevant to the user query. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {company_name}:\n{company_info} "
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            
            "Gather recent news and events about the company - {company_name}. "
            "Identify any recent developments, announcements, or changes that may impact the company's operations or strategy. "
            "This information is crucial for understanding the company's current position and future outlook."
        )    
    },
    QueryTypes.COMPANALYSIS.value: {
        "search": (
            "Gather information about the competitors of the company - {company_name}. "
            "Identify key players in the industry and their market positions. "
            "Use websites like G2 or Crunchbase to find competitors. "
            "Analyze their strengths, weaknesses, and strategies to understand the competitive landscape."
        ),
        "system": (
            "You are a competitive analyst. "
            "You need to gather information about the competitors of the company that is relevant to the user query. "
            "The information should be accurate and ONLY from the content provided. "
            "DO NOT make up any information."
        ),
        "user": (
            "Below is the information about {company_name}:\n{company_info} "
            "You are provided with content of the webpage: {webpage}. "
            "\n---CONTENT---\n{content}\n"
            "\n---END CONTENT---\n"
            
            "Gather information about the competitors of the company - {company_name}. "
            "Identify key players in the industry and their market positions. "
            "Analyze their strengths, weaknesses, and strategies to understand the competitive landscape."
        )
    }
}


def search_query(seller, company_name, query_type, history=None) -> SearchResponse:
    
    condition_dict = {
        "seller": seller, 
        "company_name": company_name, 
        "query_type": query_type
    }
    print("Checking if record exists in the database...")
    if sqldb.check_record_exists("search_results", condition_dict):
        print("Record exists, fetching from the database...")
        return SearchResponse(**sqldb.get_record("search_results", condition_dict))
    
    print("Record does not exist, making API call...")
    
    query = query_type_prompts[query_type]['search'].format(company_name=company_name)
    
    if history is None:
        history = [
            ["human", "Hi, how are you?"],
            ["assistant", "I am doing well, how can I help you today?"]
        ]
    
    headers = {"Content-Type": "application/json"}
    payload = {
        "chatModel": {
            "provider": "openai",
            "name": "gpt-4o-mini"
        },
        "embeddingModel": {
            "provider": "openai",
            "name": "text-embedding-3-large"
        },
        "optimizationMode": "speed",
        "focusMode": "webSearch",
        "query": query,
        "history": history
    }
    
    response = requests.post(API_URL, headers=headers, data=json.dumps(payload)).json()
    response_obj = SearchResponse(**{
        **response,
        "company_name": company_name,
        "seller": seller,
        "query_type": query_type,
        "query": query
    })
    
    sqldb.insert_record(
        "search_results",
        {
            "company_name": company_name,
            "seller": seller,
            "query_type": query_type,
            "query": query,
            "message": response['message'],
            "sources": response['sources'],
        }
    )
    return response_obj


def get_cited_sources(source):
    pattern = r'\[([^\]]+)\]'
    matches: List[str] = re.findall(pattern, source)
    return list(set([int(i) for i in matches if i.isnumeric()]))


def get_cited_content(sources: List[Source], citations: List[int]) -> List[CitedSource]:
    cited_content = []
    for citation in citations:
        content = sources[citation].pageContent
        title = sources[citation].metadata.title
        url = sources[citation].metadata.url
        cited_content.append(CitedSource(
            citation_id=citation,
            title=title,
            content=content,
            url=url
        ))
    return cited_content


def extract_data_from_sources(search_response: SearchResponse) -> SearchResponse:
    condition_dict = {
        "seller": search_response.seller, 
        "company_name": search_response.company_name, 
        "query_type": search_response.query_type
    }
    print("Checking if record exists in the database...")
    if sqldb.check_record_exists("search_results", condition_dict):
        print("Record exists, fetching from the database...")
        record = sqldb.get_record("search_results", condition_dict)
        if record['source_extracted_data']:
            return SearchResponse(**record)
    
    print("Record does not exist, making API call...")
    citations = get_cited_sources(search_response.message)
    if not citations:
        data = []
    else:
        cited_sources = get_cited_content(search_response.sources, citations)
        links = [source.url for source in cited_sources]
        company_info = search_response.message
        system_prompt = query_type_prompts[search_response.query_type]['system']
        user_prompt = query_type_prompts[search_response.query_type]['user'].format(
            company_name=search_response.company_name, company_info=company_info,
            webpage="{webpage}", content="{content}"
        )
        data = extract_data_from_links(links, user_prompt=user_prompt, system_prompt=system_prompt)
        extracted_data_map = {source['link']: source['data'] for source in data}
        extracted_cited_sources = [
            ExtractedCitedSource(
                citation_id=citation.citation_id,
                title=citation.title,
                content=citation.content,
                url=citation.url,
                data=extracted_data_map[citation.url]
            )
            for citation in cited_sources
            if citation.url in extracted_data_map
        ]
    
    sqldb.update_record(
        "search_results",
        condition_dict,
        {"source_extracted_data": [d.model_dump(mode='json') for d in extracted_cited_sources]}
    )
    
    return search_response.model_copy(update={"source_extracted_data": extracted_cited_sources})

company_name = "Manpower group"
seller = 'Whatfix'

### Company Overview - landing page for buyer and seller in separate indexes. 

In [34]:
search_response = search_query(
    seller=seller,
    company_name=company_name, 
    query_type=QueryTypes.FMOD.value
)

Checking if record exists in the database...
Record exists, fetching from the database...


In [35]:
print(search_response.message)

ManpowerGroup Inc. is a prominent player in the workforce solutions and staffing industry, providing a wide range of services that include recruitment, assessment, upskilling, reskilling, training and development, career management, outsourcing, and workforce consulting. Below is a detailed overview of the company's financial information, size, and industry context.

## Company Overview

### Industry
ManpowerGroup operates within the staffing and workforce solutions industry, which is characterized by the provision of temporary and permanent staffing services across various sectors. The company is recognized as one of the largest staffing firms globally, ranking third behind Adecco and Randstad as of 2020[4]. The industry is essential for connecting employers with potential employees, particularly in a rapidly changing job market.

### Size
ManpowerGroup has a significant global presence, with approximately 240 offices in the United States and Canada, and 15 offices in Europe[5]. The c

In [36]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...


### Strategic Initiatives - this should be part of landing page for buyer

In [37]:
search_response = search_query(seller=seller, company_name=company_name, query_type=QueryTypes.STRATEGY.value)

Checking if record exists in the database...
Record exists, fetching from the database...


In [38]:
print(search_response.message)

To analyze ManpowerGroup Inc.'s key business priorities, strategic initiatives, and overall financial health, we can draw insights from their 10-K reports, financial statements, and other relevant sources. This comprehensive overview will cover their operations, products, services, market position, and future plans, while also identifying challenges that the company faces.

## Key Business Priorities and Strategic Initiatives

### 1. **Diversification of Services**
ManpowerGroup is focused on diversifying its service offerings to enhance higher-margin solutions. This includes expanding its capabilities in areas such as outsourcing and workforce management, which are increasingly critical in a competitive labor market[4]. The company aims to leverage technology to improve productivity and efficiency across its operations, ensuring they remain agile and responsive to client needs[4].

### 2. **Investment in Technology**
The integration of advanced technology is a significant priority for

In [39]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...


### Recent News

In [40]:
search_response = search_query(seller=seller, company_name=company_name, query_type=QueryTypes.RECENTNEWS.value)
print(search_response.message)

Checking if record exists in the database...
Record exists, fetching from the database...
ManpowerGroup, a leading human resource consulting firm, has been active in recent months, with several developments that could significantly impact its operations and strategic direction. Here’s a summary of the latest news and events surrounding the company:

## Recent Financial Performance

### Fourth Quarter 2024 Results
On January 30, 2025, ManpowerGroup reported its fourth-quarter earnings, revealing a net earnings figure of $0.47 per diluted share for the three months ending December 31, 2024. This marked a notable recovery from a net loss of $1.73 per share in the same quarter the previous year. However, the earnings were negatively impacted by 20 cents due to fluctuations in foreign currencies compared to the prior year[1][4]. This financial performance indicates a potential rebound, although the company remains cautious about future market conditions.

### Third Quarter 2024 Insights
In 

In [41]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...


### Competitors

In [42]:
search_response = search_query(seller=seller, company_name=company_name, query_type=QueryTypes.COMPANALYSIS.value)
print(search_response.message)

Checking if record exists in the database...
Record exists, fetching from the database...
ManpowerGroup is a prominent player in the staffing and human resources industry, but it faces competition from several key players. Understanding these competitors, their market positions, strengths, weaknesses, and strategies is essential for grasping the competitive landscape. Below is a detailed analysis of ManpowerGroup's main competitors.

## Key Competitors of ManpowerGroup

### 1. **Randstad**
- **Market Position**: Randstad is one of the largest staffing firms globally, headquartered in the Netherlands. It operates in over 38 countries and has a strong presence in Europe and North America.
- **Strengths**:
  - Extensive global network and brand recognition.
  - Diverse service offerings, including staffing, recruitment, and workforce management solutions.
  - Strong technology integration in recruitment processes.
- **Weaknesses**:
  - High operational costs due to its large scale.
  - Vu

In [43]:
extracted_sources_response = extract_data_from_sources(search_response)

Checking if record exists in the database...
Record exists, fetching from the database...
